# 电子邮件主题生成器

## 练习目标（理念）

用 **OpenAI 兼容客户端**经 **OpenRouter** 读邮件正文，自动建议主题行，并判断紧急程度。

- **输入**：一封邮件正文（`email_body`）
- **输出**：恰好 3 条简洁主题行 + `Low` / `Medium` / `High` 紧急度
- **额外要求**：主题每条少于 8 个英文词、无 emoji、无多余解说

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Environment Variables / `.env` | `load_dotenv` + `OPENROUTER_API_KEY` |
| Chat Completions | `client.chat.completions.create(...)` |
| `messages`（system / user） | system 定格式约束，user 放邮件正文 |
| 兼容网关 | `base_url=https://openrouter.ai/api/v1` |

## 怎么跑

1. 在 `.env` 配置 `OPENROUTER_API_KEY`
2. 从上到下运行；可按需改 `email_body`
3. 看打印出的 Subject Options 与 Urgency


In [13]:
# ========== 导入：环境变量、笔记本展示、OpenAI 兼容客户端 ==========

# 导入标准库 os：用 getenv 读取 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量，避免写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown、display：需要时在笔记本里渲染 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI：本练习通过 OpenRouter 的兼容端点调用模型
from openai import OpenAI


## 第 1 步：导入所需库

为后续 API 通信与环境变量（Environment Variables）管理准备依赖：`os`、`dotenv`、`OpenAI`。


In [14]:
# ========== 加载 .env 并校验 OpenRouter API Key ==========

# override=True：.env 中的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境读取 OpenRouter 密钥（名字必须是 OPENROUTER_API_KEY）
api_key = os.getenv('OPENROUTER_API_KEY')
# OpenRouter 的 OpenAI 兼容 API 根地址（不要改路径形态）
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# 检查钥匙：下面 print 文案影响排查流程，保持英文原样

if not api_key:
    # 完全没读到密钥
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    # 有值但前缀不像预期（OpenRouter 密钥形态可能不同，此检查来自原作者模板）
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    # 首尾有空格/制表符，容易导致鉴权失败
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 初步看起来可用
    print("API key found and looks good so far!")


An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


## 步骤 2：加载 API 凭证并验证配置

- 用 `load_dotenv` 从 `.env` 加载环境变量
- 读取 `OPENROUTER_API_KEY`
- 做有无 / 前缀 / 首尾空白的简单校验（print 提示保持原英文）


In [15]:
# ========== 示例邮件正文：改这里即可换场景 ==========

# 例如，对于电子邮件正文：三引号字符串发给模型，英文内容不翻译

email_body = """
Hi Team,

We’ve completed the initial testing phase for the payment gateway integration. 
Most core flows are working correctly, including checkout, refunds, and invoice generation.

However, we identified two issues related to webhook delays and currency conversion rounding errors. 
The backend team is investigating, and we expect fixes by Thursday.

Please review the attached test report and share any additional feedback by tomorrow EOD.

Best,
Pranav
"""


## 步骤 3：定义示例电子邮件正文

一封讨论支付网关测试结果与问题的专业邮件示例；练习时只需改 `email_body` 字符串。


In [16]:
# ========== system / user prompt：约束输出格式（英文指令不翻译）==========

# system_prompt：角色 + 生成 3 条主题 + 紧急度 + 严格输出模板
system_prompt = """
You are an AI email assistant built for a professional email platform.

Your task:
- Read the provided email body.
- Generate exactly 3 concise, professional subject line suggestions.
- Each subject line must be under 8 words.
- Do not use emojis.
- Do not add extra commentary.
- Avoid generic phrases like "Regarding" or "Update".
- Make the subjects specific and meaningful.

Then classify the urgency of the email as:
Low, Medium, or High.

Return your output STRICTLY in this format:

Subject Options:
- <subject 1>
- <subject 2>
- <subject 3>

Urgency: <Low/Medium/High>

"""

# user_prompt：把邮件正文塞进 Email Body 区块
user_prompt = f"""
Email Body:
{email_body}
"""


## 步骤 4：创建系统和用户提示

**系统提示（system）：** 约束主题行形态与紧急度分类：
- 恰好 3 条、每条少于 8 个英文词
- 专业语气、无 emoji、避免空泛套话
- 输出必须贴给定模板

**用户提示（user）：** 只提供待分析的邮件正文


In [17]:
# ========== 组装 messages：Chat Completions 标准两角色列表 ==========

# system 在前定规矩，user 在后给邮件；顺序影响模型如何解读任务
messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]


## 步骤 5：构建消息格式

把 system / user 提示合并为标准 OpenAI `messages` 列表，供下一步 `create` 使用。


In [20]:
# ========== 经 OpenRouter 调用模型并打印结果 ==========

# 创建客户端：密钥 + OpenRouter 兼容 base_url
client = OpenAI(api_key=api_key, base_url=OPENROUTER_BASE_URL)
# chat.completions.create：model 用 gpt-4.1-nano；messages 为上一步列表
response = client.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 取出第一条 choice 的助手文本并打印（主题选项 + Urgency）
print(response.choices[0].message.content)


Subject Options:
- Payment Gateway Testing Completion
- Webhook and Currency Conversion Issues
- Request for Feedback on Test Report

Urgency: Medium


## 步骤 6：调用 OpenRouter API 并显示结果

- 用 OpenRouter 凭据初始化 `OpenAI` 客户端（`base_url` 指向兼容网关）
- 向 `gpt-4.1-nano` 发送请求
- 提取并打印主题行建议与紧急程度分类
